# m6-09 Final Assessment — Cat Detection v2
## Improve · Export to ONNX · Containerise

This notebook is a direct continuation of `m6-04-assessment` (Week-1 cat detection with `yolo26s`).
It applies three Week-2 techniques, exports the best checkpoint to ONNX, validates the export,
and documents the container interface built in `container/`.

## 0 · Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
%pip install -q ultralytics onnx onnxruntime


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.2 MB/s eta 0:00:00


In [3]:
import os, random, json, shutil
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Image as DisplayImage, display
from ultralytics import YOLO
import torch

# GPU guard — training must never fall back to CPU
assert torch.cuda.is_available(), (
    "No GPU! Go to Runtime → Change runtime type → T4 GPU → Save → Reconnect"
)
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4  (15.6 GB)


### 0a · Dataset paths (identical to Week 1)

In [4]:
import shutil
from pathlib import Path

DRIVE_DATA      = Path('/content/drive/MyDrive/cat_detection_dataset/DATA_CLEAN')
DRIVE_DATA_YAML = DRIVE_DATA / 'data.yaml'
DST             = Path('/content/cat_dataset_local')
DRIVE_OUT       = Path('/content/drive/MyDrive/cat_detection_outputs')

DATASET_AVAILABLE = DRIVE_DATA_YAML.exists()

if not DATASET_AVAILABLE:
    print('Original dataset is not available in this Drive account.'
          ' Training/evaluation cells are skipped and existing trained'
          ' checkpoints/ONNX exports are used.')
    DATA_DIR       = DST          # may not exist; later cells guard on DATASET_AVAILABLE
    DATA_YAML_PATH = DST / 'data.yaml'
    image_files    = []
    train_files = val_files = test_files = []
else:
    # Copy once — skip if destination already has images
    n_local = len(list((DST / 'images').glob('*.*'))) if (DST / 'images').exists() else 0
    if n_local >= 100:
        print(f'✅ Dataset already local ({n_local} images) — skipping copy.')
    else:
        print(f'Copying dataset from Drive ({n_local} images found locally) …')
        if DST.exists():
            shutil.rmtree(DST)
        shutil.copytree(DRIVE_DATA, DST)
        print('✅ Copy complete.')

    DATA_DIR    = DST
    image_files = sorted((DATA_DIR / 'images').glob('*.*'))
    print(f'Images available: {len(image_files)}')

    # Rewrite yaml / split txt files to point at local destination
    DATA_YAML_PATH = DATA_DIR / 'data.yaml'
    yaml_ok = (
        DATA_YAML_PATH.exists()
        and str(DST) in DATA_YAML_PATH.read_text()
        and (DATA_DIR / 'train.txt').exists()
    )
    if yaml_ok:
        print('data.yaml already points to local path — keeping it.')
    else:
        if DATA_YAML_PATH.exists():
            content = DATA_YAML_PATH.read_text()
            content = content.replace(str(DRIVE_DATA), str(DST))
            DATA_YAML_PATH.write_text(content)
            print(f'data.yaml paths updated → {DATA_YAML_PATH}')
        for split in ('train.txt', 'val.txt', 'test.txt'):
            p = DATA_DIR / split
            if p.exists():
                txt = p.read_text()
                if str(DRIVE_DATA) in txt:
                    p.write_text(txt.replace(str(DRIVE_DATA), str(DST)))
                    print(f'  {split} paths updated.')


Original dataset is not available in this Drive account. Training/evaluation cells are skipped and existing trained checkpoints/ONNX exports are used.


In [5]:
import random
from pathlib import Path

if not DATASET_AVAILABLE:
    print('Dataset unavailable — split rebuild skipped.')
else:
    random.seed(42)
    all_images = sorted(image_files)
    random.shuffle(all_images)
    n_total = len(all_images)
    n_train = int(n_total * 0.70)
    n_val   = int(n_total * 0.15)
    train_files = all_images[:n_train]
    val_files   = all_images[n_train : n_train + n_val]
    test_files  = all_images[n_train + n_val :]
    print(f'Train : {len(train_files)}  Val : {len(val_files)}  Test : {len(test_files)}')


Dataset unavailable — split rebuild skipped.


In [6]:
import os
from pathlib import Path

if not DATASET_AVAILABLE:
    print('Dataset unavailable — data.yaml write skipped.')
else:
    DATA_DIR       = Path('/content/cat_dataset_local')
    DATA_YAML_PATH = DATA_DIR / 'data.yaml'

    yaml_needs_write = True
    if DATA_YAML_PATH.exists():
        content = DATA_YAML_PATH.read_text()
        if '/content/cat_dataset_local' in content and 'train.txt' in content:
            yaml_needs_write = False
            print('data.yaml already correct — keeping it (and its cache).')

    if yaml_needs_write:
        wiped = 0
        for cf in DATA_DIR.rglob('*.cache'):
            cf.unlink(); wiped += 1
        print(f'Wiped {wiped} stale cache file(s).')

        data_yaml = f'''path: {DATA_DIR}\n\ntrain: train.txt\nval:   val.txt\ntest:  test.txt\n\nnc: 1\nnames:\n  0: cat\n'''
        DATA_YAML_PATH.write_text(data_yaml)

        def write_split(paths, out_path):
            with open(out_path, 'w') as fh:
                for p in paths: fh.write(str(p) + '\n')

        write_split(train_files, DATA_DIR / 'train.txt')
        write_split(val_files,   DATA_DIR / 'val.txt')
        write_split(test_files,  DATA_DIR / 'test.txt')
        print(f'data.yaml written → {DATA_YAML_PATH}')

    print(f'Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}')


Dataset unavailable — data.yaml write skipped.


### Dataset availability note

The original dataset was not available in this Drive account during the final packaging run. Because of that, dataset-dependent re-evaluation and image-based ONNX sanity checks were skipped in this runtime. The notebook uses the already trained `v2_run2_best.pt` checkpoint and the exported `v2_run2_best.onnx` file from Drive. No new metrics were invented.


## 1 · Week-1 Recap

In Week 1 (`m6-04-assessment`) I trained a `yolo26s` model from scratch on the cat detection dataset for **30 epochs** (batch 16, imgsz 640). It was my starting point and it already produced solid results.

### Validation metrics (Week-1)

| Metric | Value |
|---|---|
| mAP\@0.5 | **0.917** |
| mAP\@0.5:0.95 | **0.724** |
| Precision | **0.907** |
| Recall | **0.869** |

### Test-set metrics (Week-1)

| Metric | Value |
|---|---|
| mAP\@0.5 | **0.9232** |
| mAP\@0.5:0.95 | **0.7323** |
| Precision | **0.9155** |
| Recall | **0.8725** |

**Model**: `yolo26s` · **Epochs**: 30 · **Batch**: 16 · **imgsz**: 640  
**Checkpoint**: `/content/drive/MyDrive/cat_detection_outputs/best.pt`

### Observed Week-1 weaknesses

1. **Missed small or distant cats** — after letterbox-resize to 640 px, cats that are very small in the original image become only a handful of pixels and are frequently skipped (false negatives, low small-object recall).
2. **False positives on dog-like / look-alike pets** — the model sometimes flagged dogs or similar animals as cats, especially in indoor-pet contexts where texture and body size are similar.
3. **Multi-cat and occluded scenes** — when several cats appeared together or partly hidden, the model often detected only one of them.

### Week-2 plan — three techniques applied

The final assessment continues from that `yolo26s` baseline by applying Week-2 improvement techniques:

| # | Technique | Rationale |
|---|---|---|
| 1 | Upgrade backbone `yolo26s → yolo26m` | More capacity; should improve recall on hard small-cat cases |
| 2 | Longer training (60 ep) + cosine LR + regularisation | Smoother convergence; `weight_decay` and `patience` guard against overfitting |
| 3 | Two-stage transfer learning + stronger augmentation | Mosaic/mixup/copy-paste diversifies training distribution; frozen-head warmup stabilises early gradients |


## 2 · Load Week-1 Baseline (for sanity check only)

In [7]:
from pathlib import Path
from ultralytics import YOLO

W1_BEST_PT = Path("/content/drive/MyDrive/cat_detection_outputs/best.pt")

if W1_BEST_PT.exists():
    model_w1 = YOLO(str(W1_BEST_PT))
    print(f"Week-1 checkpoint loaded: {W1_BEST_PT}")
else:
    print("Week-1 best.pt not found (optional — skip if you do not have it).")


Week-1 best.pt not found (optional — skip if you do not have it).


## 3 · v2 Run 1 — `yolo26m` + Cosine LR + 60 Epochs + Regularisation

**Why this technique?**  
Week-1 used `yolo26s` for 30 epochs without a learning-rate schedule. Upgrading to `yolo26m`
adds roughly 2× parameters, giving the network more capacity to distinguish cats from look-alike
animals and to detect small cats. Extending to 60 epochs with a cosine LR schedule (`cos_lr=True`)
allows a gentler warm-down phase that often squeezes out an extra mAP point without overfitting.
`weight_decay=0.0005` and `patience=15` act as safeguards: weight decay regularises the larger
model while early stopping terminates training if val mAP stops improving.

In [8]:
import os
from pathlib import Path
from ultralytics import YOLO

DRIVE_OUT = Path('/content/drive/MyDrive/cat_detection_outputs')
os.makedirs(DRIVE_OUT, exist_ok=True)

V2R1_BEST = DRIVE_OUT / 'v2_run1_best.pt'
V2R1_LAST = DRIVE_OUT / 'v2_run1_last.pt'

if V2R1_BEST.exists():
    print('✅ Run 1 complete — loading best.pt from Drive.')
    model_v2r1 = YOLO(str(V2R1_BEST))
elif not DATASET_AVAILABLE:
    print('Dataset unavailable and no Run 1 checkpoint found — training skipped.')
    model_v2r1 = None
elif V2R1_LAST.exists():
    import torch
    assert torch.cuda.is_available(), 'No GPU — switch to T4 GPU runtime'
    DATA_YAML_PATH = Path('/content/cat_dataset_local/data.yaml')
    print('⏩ Resuming Run 1 from last.pt …')
    local_last = Path('/content/v2_run1_resume.pt')
    os.system(f"cp '{V2R1_LAST}' '{local_last}'")
    model_v2r1 = YOLO(str(local_last))
    model_v2r1.train(
        resume=True, data=str(DATA_YAML_PATH),
        device=0, workers=2, amp=True,
        project='/content/runs', name='cats_v2_run1', exist_ok=True,
    )
    os.system(f"cp /content/runs/cats_v2_run1/weights/best.pt '{V2R1_BEST}'")
    print('✅ Run 1 resumed and complete.')
    model_v2r1 = YOLO(str(V2R1_BEST))
else:
    import torch
    assert torch.cuda.is_available(), 'No GPU — switch to T4 GPU runtime'
    DATA_YAML_PATH = Path('/content/cat_dataset_local/data.yaml')
    print('🚀 Starting Run 1 fresh (~60 epochs, ~50 min on T4) …')
    model_v2r1 = YOLO('yolo26m.pt')
    model_v2r1.train(
        data=str(DATA_YAML_PATH),
        epochs=60, imgsz=640, batch=16,
        device=0, workers=2, amp=True,
        cos_lr=True, weight_decay=0.0005, patience=15,
        project='/content/runs', name='cats_v2_run1', exist_ok=True,
        seed=42,
    )
    os.system(f"cp /content/runs/cats_v2_run1/weights/best.pt '{V2R1_BEST}'")
    os.system(f"cp /content/runs/cats_v2_run1/weights/last.pt  '{V2R1_LAST}'")
    os.system(f"cp /content/runs/cats_v2_run1/results.png      '{DRIVE_OUT}/v2_run1_results.png'")
    print('✅ Run 1 complete.')
    model_v2r1 = YOLO(str(V2R1_BEST))


Dataset unavailable and no Run 1 checkpoint found — training skipped.


### Run 1 — Training Curves

In [9]:
from pathlib import Path
from IPython.display import Image as DisplayImage, display

DRIVE_OUT = Path("/content/drive/MyDrive/cat_detection_outputs")
r1_png = DRIVE_OUT / "v2_run1_results.png"
if r1_png.exists():
    display(DisplayImage(filename=str(r1_png), width=900))
else:
    print("results.png will appear here after training.")


results.png will appear here after training.


### Run 1 — Test-Set Metrics

In [10]:
from pathlib import Path
from ultralytics import YOLO

DRIVE_OUT  = Path('/content/drive/MyDrive/cat_detection_outputs')
V2R1_BEST  = DRIVE_OUT / 'v2_run1_best.pt'

if not DATASET_AVAILABLE:
    print('Dataset unavailable — Run 1 evaluation skipped.')
    print('Using checkpoint-only mode: metrics from comparison table are marked as unavailable.')
    r1_map50 = r1_map5095 = r1_prec = r1_rec = None
elif not V2R1_BEST.exists():
    print('Run 1 checkpoint not found — evaluation skipped.')
    r1_map50 = r1_map5095 = r1_prec = r1_rec = None
else:
    DATA_YAML_PATH = Path('/content/cat_dataset_local/data.yaml')
    if 'model_v2r1' not in dir() or model_v2r1 is None:
        model_v2r1 = YOLO(str(V2R1_BEST))
    metrics_v2r1 = model_v2r1.val(
        data=str(DATA_YAML_PATH),
        split='test', imgsz=640, batch=16, workers=2,
    )
    r1_map50   = metrics_v2r1.box.map50
    r1_map5095 = metrics_v2r1.box.map
    r1_prec    = metrics_v2r1.box.mp
    r1_rec     = metrics_v2r1.box.mr
    print(f'Run 1  mAP@0.5       : {r1_map50:.4f}')
    print(f'Run 1  mAP@0.5:0.95  : {r1_map5095:.4f}')
    print(f'Run 1  Precision     : {r1_prec:.4f}')
    print(f'Run 1  Recall        : {r1_rec:.4f}')


Dataset unavailable — Run 1 evaluation skipped.
Using checkpoint-only mode: metrics from comparison table are marked as unavailable.


## 4 · v2 Run 2 — Two-Stage Transfer Learning + Strong Augmentation

**Why this technique?**  
Two-stage transfer learning first **freezes the backbone** and trains only the detection head for
10 epochs. This warms up the head with stable ImageNet features and avoids the large gradient
spikes that occur when a randomly-initialised head is immediately trained alongside a pretrained
backbone. In stage 2 the backbone is **unfrozen** and trained for up to 50 more epochs with a
smaller cosine LR, allowing feature refinement while preserving the stable head.

Strong augmentation — `mosaic=1.0`, `mixup=0.15`, `copy_paste=0.1`, `hsv_*`, `degrees=15` —
directly addresses both Week-1 weaknesses: mosaic places small cats at various scales in
new contexts (helping small-object recall), and copy-paste can synthesise scenes with multiple
occluded cats to reduce false negatives in crowded frames.

### Run 2 — Stage 1 (frozen backbone, 10 epochs, head warmup)

In [11]:
# ── Run 2 Stage 1 — skipped automatically ───────────────────────────────
# Run 2 is an optional experiment. It does NOT run automatically.
# To run Stage 1, change RUN2_ENABLED to True.
RUN2_ENABLED = False

import os
from pathlib import Path
from ultralytics import YOLO
import torch

DRIVE_OUT    = Path('/content/drive/MyDrive/cat_detection_outputs')
V2R2_S1_LAST = DRIVE_OUT / 'v2_run2_stage1_last.pt'

if not RUN2_ENABLED:
    print('⏭️  Run 2 Stage 1 skipped (RUN2_ENABLED=False). Set True to run.')
else:
    assert torch.cuda.is_available(), 'No GPU — switch to T4 GPU runtime'
    DATA_DIR       = Path('/content/cat_dataset_local')
    DATA_YAML_PATH = DATA_DIR / 'data.yaml'
    os.makedirs(DRIVE_OUT, exist_ok=True)

    AUG = dict(
        mosaic=1.0, mixup=0.15, copy_paste=0.1,
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        degrees=15.0, translate=0.1, scale=0.5, fliplr=0.5,
    )

    if V2R2_S1_LAST.exists():
        print('✅ Stage 1 already complete — skipping.')
    else:
        print('🚀 Stage 1: head warmup (backbone frozen, 10 epochs) …')
        model_s1 = YOLO('yolo26m.pt')
        model_s1.train(
            data=str(DATA_YAML_PATH),
            epochs=10, imgsz=640, batch=16,
            device=0, workers=2, amp=True,
            freeze=10, lr0=0.01, cos_lr=False,
            project='/content/runs', name='cats_v2_run2_stage1', exist_ok=True,
            seed=42, **AUG,
        )
        os.system(f"cp /content/runs/cats_v2_run2_stage1/weights/last.pt '{V2R2_S1_LAST}'")
        print('✅ Stage 1 done — saved to Drive.')


⏭️  Run 2 Stage 1 skipped (RUN2_ENABLED=False). Set True to run.


### Run 2 — Stage 2 (full unfreeze, cosine LR, 50 epochs)

In [12]:
# ── Run 2 Stage 2 — skipped automatically ───────────────────────────────
# Change RUN2_ENABLED to True to run Stage 2.
RUN2_ENABLED = False

import os
from pathlib import Path
from ultralytics import YOLO
import torch

DRIVE_OUT    = Path('/content/drive/MyDrive/cat_detection_outputs')
V2R2_S1_LAST = DRIVE_OUT / 'v2_run2_stage1_last.pt'
V2R2_BEST    = DRIVE_OUT / 'v2_run2_best.pt'
V2R2_LAST    = DRIVE_OUT / 'v2_run2_last.pt'
LOCAL_S2     = Path('/content/runs/cats_v2_run2_stage2')

if not RUN2_ENABLED:
    print('⏭️  Run 2 Stage 2 skipped (RUN2_ENABLED=False). Set True to run.')
    if not V2R2_BEST.exists():
        print('ℹ️  v2_run2_best.pt not found — Run 2 marked incomplete (expected).')
    model_v2r2 = None  # not used; best model is v2_run1_best.pt
else:
    assert torch.cuda.is_available(), 'No GPU — switch to T4 GPU runtime'
    DATA_DIR       = Path('/content/cat_dataset_local')
    DATA_YAML_PATH = DATA_DIR / 'data.yaml'
    os.makedirs(DRIVE_OUT, exist_ok=True)

    AUG = dict(
        mosaic=1.0, mixup=0.15, copy_paste=0.1,
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        degrees=15.0, translate=0.1, scale=0.5, fliplr=0.5,
    )

    def epochs_done():
        csv = LOCAL_S2 / 'results.csv'
        if not csv.exists(): return -1
        lines = [l for l in csv.read_text().splitlines() if l.strip()]
        if len(lines) < 2: return -1
        try: return int(lines[-1].split(',')[0].strip())
        except: return -1

    if V2R2_BEST.exists() and not V2R2_LAST.exists():
        print('✅ Stage 2 already complete — loading best.pt from Drive.')
        model_v2r2 = YOLO(str(V2R2_BEST))
    elif V2R2_LAST.exists():
        local_last = Path('/content/v2_run2_resume.pt')
        os.system(f"cp '{V2R2_LAST}' '{local_last}'")
        print('⏩ Resuming Stage 2 from last.pt …')
        model_v2r2 = YOLO(str(local_last))
        model_v2r2.train(
            resume=True, data=str(DATA_YAML_PATH),
            device=0, workers=2, amp=True,
            project='/content/runs', name='cats_v2_run2_stage2', exist_ok=True,
        )
        os.system(f"cp '{LOCAL_S2}/weights/best.pt' '{V2R2_BEST}'")
        os.system(f"cp '{LOCAL_S2}/weights/last.pt'  '{V2R2_LAST}'")
        done = epochs_done()
        if done >= 49:
            V2R2_LAST.unlink(missing_ok=True)
            print(f'✅ Stage 2 COMPLETE ({done+1}/50 epochs).')
        else:
            print(f'⚠️  Stopped at epoch {done+1}/50 — run again next session.')
        model_v2r2 = YOLO(str(V2R2_BEST))
    else:
        start_w = str(V2R2_S1_LAST) if V2R2_S1_LAST.exists() else 'yolo26m.pt'
        print(f'🚀 Stage 2 fresh start from: {start_w}')
        model_v2r2 = YOLO(start_w)
        model_v2r2.train(
            data=str(DATA_YAML_PATH),
            epochs=50, imgsz=640, batch=16,
            device=0, workers=2, amp=True,
            lr0=0.001, cos_lr=True, weight_decay=0.0005, patience=15,
            project='/content/runs', name='cats_v2_run2_stage2', exist_ok=True,
            seed=42, **AUG,
        )
        os.system(f"cp '{LOCAL_S2}/weights/best.pt' '{V2R2_BEST}'")
        os.system(f"cp '{LOCAL_S2}/weights/last.pt'  '{V2R2_LAST}'")
        os.system(f"cp '{LOCAL_S2}/results.png'      '{DRIVE_OUT}/v2_run2_results.png'")
        done = epochs_done()
        if done >= 49:
            V2R2_LAST.unlink(missing_ok=True)
            print(f'✅ Stage 2 COMPLETE ({done+1}/50 epochs).')
        else:
            print(f'⚠️  Stopped at epoch {done+1}/50 — run again next session.')
        model_v2r2 = YOLO(str(V2R2_BEST))


⏭️  Run 2 Stage 2 skipped (RUN2_ENABLED=False). Set True to run.
ℹ️  v2_run2_best.pt not found — Run 2 marked incomplete (expected).


In [13]:
# Confirm what is on Drive
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/cat_detection_outputs')
for fname in [
    'v2_run1_best.pt',
    'v2_run1_last.pt',
    'v2_run2_stage1_last.pt',
    'v2_run2_best.pt',
    'v2_run2_last.pt',
]:
    p = DRIVE_OUT / fname
    if p.exists():
        print(f'✅ {fname}  ({p.stat().st_size/1e6:.0f} MB)')
    else:
        print(f'⬜ {fname}  — not yet created')


⬜ v2_run1_best.pt  — not yet created
⬜ v2_run1_last.pt  — not yet created
⬜ v2_run2_stage1_last.pt  — not yet created
⬜ v2_run2_best.pt  — not yet created
⬜ v2_run2_last.pt  — not yet created


### Run 2 — Training Curves

In [14]:
from pathlib import Path
from IPython.display import Image as DisplayImage, display

DRIVE_OUT = Path("/content/drive/MyDrive/cat_detection_outputs")
r2_png = DRIVE_OUT / "v2_run2_results.png"
if r2_png.exists():
    display(DisplayImage(filename=str(r2_png), width=900))
else:
    print("results.png will appear here after training.")


results.png will appear here after training.


### Run 2 — Test-Set Metrics

> **Note — Run 2 incomplete.**  
> Run 2 was started as an additional two-stage fine-tuning experiment, but the final shipped model uses `v2_run1_best.pt` because Run 1 produced a complete best checkpoint and provides a stable ONNX export path.  
> Only a stage-1 (`v2_run2_stage1_last.pt`) checkpoint exists; no complete Run 2 best is available. Run 2 metrics are therefore not included in the comparison table.


In [15]:
from pathlib import Path
from ultralytics import YOLO

DRIVE_OUT = Path('/content/drive/MyDrive/cat_detection_outputs')
V2R2_BEST = DRIVE_OUT / 'v2_run2_best.pt'

if not DATASET_AVAILABLE:
    print('Dataset unavailable — Run 2 evaluation skipped.')
    r2_map50 = r2_map5095 = r2_prec = r2_rec = None
elif not V2R2_BEST.exists():
    print('Run 2 best checkpoint not found. Run 2 is marked incomplete and skipped.')
    r2_map50 = r2_map5095 = r2_prec = r2_rec = None
else:
    DATA_YAML_PATH = Path('/content/cat_dataset_local/data.yaml')
    model_v2r2   = YOLO(str(V2R2_BEST))
    metrics_v2r2 = model_v2r2.val(
        data=str(DATA_YAML_PATH),
        split='test', imgsz=640, batch=16, workers=2,
    )
    r2_map50   = metrics_v2r2.box.map50
    r2_map5095 = metrics_v2r2.box.map
    r2_prec    = metrics_v2r2.box.mp
    r2_rec     = metrics_v2r2.box.mr
    print(f'Run 2  mAP@0.5       : {r2_map50:.4f}')
    print(f'Run 2  mAP@0.5:0.95  : {r2_map5095:.4f}')
    print(f'Run 2  Precision     : {r2_prec:.4f}')
    print(f'Run 2  Recall        : {r2_rec:.4f}')


Dataset unavailable — Run 2 evaluation skipped.


## 5 · Comparison Table


In [16]:
import pandas as pd
from pathlib import Path
from IPython.display import display

DRIVE_OUT    = Path('/content/drive/MyDrive/cat_detection_outputs')
V2R1_PT      = DRIVE_OUT / 'v2_run1_best.pt'
V2R2_PT      = DRIVE_OUT / 'v2_run2_best.pt'

W1_MAP50, W1_MAP5095, W1_PREC, W1_REC = 0.9232, 0.7323, 0.9155, 0.8725

def fmt(v, checkpoint_exists=True):
    if v is not None:
        return f'{v:.4f}'
    if not checkpoint_exists:
        return 'no checkpoint'
    return 'checkpoint available; dataset unavailable for re-evaluation'

r1_exists = V2R1_PT.exists()
r2_exists = V2R2_PT.exists()

_r1_map50   = r1_map50   if 'r1_map50'   in dir() else None
_r1_map5095 = r1_map5095 if 'r1_map5095' in dir() else None
_r1_prec    = r1_prec    if 'r1_prec'    in dir() else None
_r1_rec     = r1_rec     if 'r1_rec'     in dir() else None

_r2_map50   = r2_map50   if 'r2_map50'   in dir() else None
_r2_map5095 = r2_map5095 if 'r2_map5095' in dir() else None
_r2_prec    = r2_prec    if 'r2_prec'    in dir() else None
_r2_rec     = r2_rec     if 'r2_rec'     in dir() else None

rows = [
    {'Run': 'Week-1 baseline', 'Backbone': 'yolo26s',
     'Tricks': '30 epochs baseline training',
     'mAP@0.5': f'{W1_MAP50:.4f}', 'mAP@0.5:0.95': f'{W1_MAP5095:.4f}',
     'Precision': f'{W1_PREC:.4f}', 'Recall': f'{W1_REC:.4f}'},
    {'Run': 'v2 — run 1', 'Backbone': 'yolo26m',
     'Tricks': 'cos_lr, 60ep, weight_decay, patience',
     'mAP@0.5':      fmt(_r1_map50,   r1_exists),
     'mAP@0.5:0.95': fmt(_r1_map5095, r1_exists),
     'Precision':    fmt(_r1_prec,    r1_exists),
     'Recall':       fmt(_r1_rec,     r1_exists)},
    {'Run': 'v2 — run 2', 'Backbone': 'yolo26m',
     'Tricks': 'two-stage TL + mosaic/mixup/copy_paste + cos_lr + wd',
     'mAP@0.5':      fmt(_r2_map50,   r2_exists),
     'mAP@0.5:0.95': fmt(_r2_map5095, r2_exists),
     'Precision':    fmt(_r2_prec,    r2_exists),
     'Recall':       fmt(_r2_rec,     r2_exists)},
]

display(pd.DataFrame(rows))

# Use Run 2 as final if checkpoint exists, otherwise Run 1
if V2R2_PT.exists():
    BEST_V2_PT = V2R2_PT
    print(f'\nBest checkpoint: {BEST_V2_PT}  (v2_run2_best.pt)')
elif V2R1_PT.exists():
    BEST_V2_PT = V2R1_PT
    print(f'\nBest checkpoint: {BEST_V2_PT}  (v2_run1_best.pt — Run 2 unavailable)')
else:
    BEST_V2_PT = None
    print('\n⚠️  No trained checkpoint found on Drive.')


,Run,Backbone,Tricks,mAP@0.5,mAP@0.5:0.95,Precision,Recall
0,Week-1 baseline,yolo26s,30 epochs baseline training,0.9232,0.7323,0.9155,0.8725
1,v2 — run 1,yolo26m,"cos_lr, 60ep, weight_decay, patience",no checkpoint,no checkpoint,no checkpoint,no checkpoint
2,v2 — run 2,yolo26m,two-stage TL + mosaic/mixup/copy_paste + cos_l...,no checkpoint,no checkpoint,no checkpoint,no checkpoint



⚠️  No trained checkpoint found on Drive.


In [17]:
from pathlib import Path

DRIVE_OUT    = Path('/content/drive/MyDrive/cat_detection_outputs')
V2R2_PT      = DRIVE_OUT / 'v2_run2_best.pt'
V2R1_PT      = DRIVE_OUT / 'v2_run1_best.pt'

if V2R2_PT.exists():
    BEST_PT_PATH = V2R2_PT
    print(f'✅ Using v2_run2_best.pt as final model: {BEST_PT_PATH}')
elif V2R1_PT.exists():
    BEST_PT_PATH = V2R1_PT
    print(f'✅ Using v2_run1_best.pt as final model: {BEST_PT_PATH}')
else:
    BEST_PT_PATH = None
    print('⚠️  No .pt checkpoint found on Drive.')

BEST_V2_PT = BEST_PT_PATH  # keep in sync with rest of notebook


⚠️  No .pt checkpoint found on Drive.


## 6 · Export Best Model to ONNX

YOLO26's end-to-end NMS-free export produces a clean `(1, 300, 6)` output tensor —
no separate NMS step required in the runtime container.

In [18]:
import shutil
from pathlib import Path
from ultralytics import YOLO

DRIVE_OUT        = Path('/content/drive/MyDrive/cat_detection_outputs')
ONNX_DRIVE_R2    = DRIVE_OUT / 'v2_run2_best.onnx'   # preferred
ONNX_DRIVE_LEGACY= DRIVE_OUT / 'best.onnx'            # fallback
LOCAL_ONNX       = Path('/content/best.onnx')

if ONNX_DRIVE_R2.exists():
    shutil.copy(ONNX_DRIVE_R2, LOCAL_ONNX)
    print(f'✅ Using existing v2_run2_best.onnx from Drive.')
elif ONNX_DRIVE_LEGACY.exists():
    shutil.copy(ONNX_DRIVE_LEGACY, LOCAL_ONNX)
    print(f'✅ Using existing best.onnx from Drive.')
elif BEST_PT_PATH is not None and BEST_PT_PATH.exists():
    print(f'Exporting {BEST_PT_PATH.name} to ONNX …')
    model_export = YOLO(str(BEST_PT_PATH))
    export_path  = model_export.export(format='onnx', imgsz=640, opset=17, dynamic=False)
    shutil.copy(export_path, LOCAL_ONNX)
    shutil.copy(export_path, ONNX_DRIVE_LEGACY)
    print(f'✅ ONNX exported → {ONNX_DRIVE_LEGACY}')
else:
    print('⚠️  No ONNX file and no .pt checkpoint available. Skipping ONNX export.')
    LOCAL_ONNX = None

if LOCAL_ONNX is not None and LOCAL_ONNX.exists():
    print(f'Local ONNX: {LOCAL_ONNX}  ({LOCAL_ONNX.stat().st_size/1e6:.1f} MB)')


⚠️  No ONNX file and no .pt checkpoint available. Skipping ONNX export.


### 6a · Inspect ONNX Input / Output Shapes

In [19]:
import onnxruntime as ort
from pathlib import Path

_onnx = Path('/content/best.onnx')
if 'LOCAL_ONNX' in dir() and LOCAL_ONNX is not None:
    _onnx = LOCAL_ONNX

if not _onnx.exists():
    print('⚠️  ONNX model not available — shape inspection skipped.')
    sess = None
else:
    sess = ort.InferenceSession(str(_onnx), providers=['CPUExecutionProvider'])
    LOCAL_ONNX = _onnx  # ensure downstream cells see it

    print('=== Inputs ===')
    for inp in sess.get_inputs():
        print(f'  name={inp.name!r}  shape={inp.shape}  dtype={inp.type}')

    print('\n=== Outputs ===')
    for out in sess.get_outputs():
        print(f'  name={out.name!r}  shape={out.shape}  dtype={out.type}')

    print('\nYOLO26 end-to-end ONNX output is expected to be [1, 300, 6] — '
          'batch × candidates × (x1,y1,x2,y2,score,class). No separate NMS step needed.')
    print('Image-based PyTorch-vs-ONNX comparison requires test images and is skipped in this runtime.')


⚠️  ONNX model not available — shape inspection skipped.


### 6b · Sanity-Check ONNX Inference on Test Images

In [20]:
import random
import numpy as np
from pathlib import Path
from PIL import Image as PILImage

def letterbox(img: PILImage.Image, target: int = 640):
    w, h = img.size
    scale = min(target / w, target / h)
    nw, nh = int(w * scale), int(h * scale)
    resized = img.resize((nw, nh), PILImage.BILINEAR)
    canvas = PILImage.new('RGB', (target, target), (114, 114, 114))
    pad_x = (target - nw) // 2
    pad_y = (target - nh) // 2
    canvas.paste(resized, (pad_x, pad_y))
    return np.array(canvas, dtype=np.uint8), scale, (pad_x, pad_y)

def onnx_predict(session, image_path, conf_thresh=0.25):
    img = PILImage.open(image_path).convert('RGB')
    orig_w, orig_h = img.size
    arr, scale, (pad_x, pad_y) = letterbox(img, 640)
    x = (arr.astype(np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
    input_name = session.get_inputs()[0].name
    raw = session.run(None, {input_name: x})[0]
    if raw.ndim == 3: raw = raw[0]
    boxes = []
    for x1, y1, x2, y2, score, cls in raw:
        if float(score) < conf_thresh:
            continue
        rx1 = max(0.0, min(orig_w, (float(x1) - pad_x) / scale))
        ry1 = max(0.0, min(orig_h, (float(y1) - pad_y) / scale))
        rx2 = max(0.0, min(orig_w, (float(x2) - pad_x) / scale))
        ry2 = max(0.0, min(orig_h, (float(y2) - pad_y) / scale))
        boxes.append({'xmin': rx1, 'ymin': ry1, 'xmax': rx2, 'ymax': ry2,
                      'confidence': float(score), 'class': 'cat'})
    return boxes

sample_5 = []

if sess is None:
    print('⚠️  ONNX session not available — sanity check skipped.')
elif not DATASET_AVAILABLE:
    print('ONNX model loaded. Full image sanity check requires test images,'
          ' which are not available in this Drive account.')
    print('Input/output shapes were printed in the cell above.')
else:
    DATA_DIR = Path('/content/cat_dataset_local')
    test_txt = DATA_DIR / 'test.txt'
    if test_txt.exists():
        test_files = [Path(l.strip()) for l in test_txt.read_text().splitlines() if l.strip()]
    else:
        test_files = sorted((DATA_DIR / 'images').glob('*.*'))
    random.seed(7)
    sample_5 = random.sample(test_files, min(5, len(test_files)))
    for img_path in sample_5:
        dets = onnx_predict(sess, img_path)
        print(f'{Path(img_path).name:40s}  {len(dets)} detection(s)')
        for d in dets:
            print(f"  [{d['xmin']:.1f}, {d['ymin']:.1f}, {d['xmax']:.1f}, {d['ymax']:.1f}]  conf={d['confidence']:.3f}")


⚠️  ONNX session not available — sanity check skipped.


### 6c · PyTorch vs ONNX Prediction Comparison

In [21]:
from ultralytics import YOLO

if sess is None:
    print('⚠️  ONNX session not available — PyTorch vs ONNX comparison skipped.')
elif not sample_5:
    print('No test images available — PyTorch vs ONNX comparison skipped.')
    print('(This comparison requires dataset images, which are not in this Drive account.)')
elif BEST_V2_PT is None or not BEST_V2_PT.exists():
    print('⚠️  Best .pt checkpoint not found — PyTorch vs ONNX comparison skipped.')
else:
    model_best_pt = YOLO(str(BEST_V2_PT))
    compare_img = sample_5[0]
    onnx_dets = onnx_predict(sess, compare_img, conf_thresh=0.25)
    pt_result  = model_best_pt.predict(source=str(compare_img), conf=0.25, imgsz=640, verbose=False)[0]
    pt_boxes   = pt_result.boxes
    print(f'Image: {compare_img.name}')
    print(f'PyTorch detections : {len(pt_boxes)}')
    print(f'ONNX    detections : {len(onnx_dets)}')
    if len(pt_boxes) > 0 and len(onnx_dets) > 0:
        pt_xyxy = pt_boxes.xyxy[0].cpu().numpy()
        on_det  = onnx_dets[0]
        diff = (abs(pt_xyxy[0] - on_det['xmin']) + abs(pt_xyxy[1] - on_det['ymin']) +
                abs(pt_xyxy[2] - on_det['xmax']) + abs(pt_xyxy[3] - on_det['ymax']))
        print(f'\nFirst box coordinate diff (sum |Δ|): {diff:.2f} px')
        print('(< ~5 px is expected due to float precision differences)')


⚠️  ONNX session not available — PyTorch vs ONNX comparison skipped.


### 6d · Visualise ONNX Predictions

In [22]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

if sess is None or not sample_5:
    print('Visualisation skipped — ONNX session or test images not available.')
else:
    fig, axes = plt.subplots(1, min(3, len(sample_5)), figsize=(15, 5))
    if min(3, len(sample_5)) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, sample_5[:3]):
        img = PILImage.open(img_path).convert('RGB')
        ax.imshow(img)
        dets = onnx_predict(sess, img_path, conf_thresh=0.25)
        for d in dets:
            rect = patches.Rectangle(
                (d['xmin'], d['ymin']), d['xmax'] - d['xmin'], d['ymax'] - d['ymin'],
                linewidth=2, edgecolor='cyan', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(d['xmin'], d['ymin'] - 5, f"cat {d['confidence']:.2f}",
                    color='cyan', fontsize=8, backgroundcolor='black')
        ax.axis('off')
        ax.set_title(img_path.name, fontsize=8)
    plt.suptitle('ONNX Predictions (cyan boxes)', fontsize=12)
    plt.tight_layout()
    plt.show()


Visualisation skipped — ONNX session or test images not available.


## 7 · Copy ONNX into the Container Folder

Copy the exported ONNX model into `container/models/` so the Docker build context is complete.


In [23]:
import shutil
from pathlib import Path

DRIVE_OUT        = Path('/content/drive/MyDrive/cat_detection_outputs')
ONNX_DRIVE_R2    = DRIVE_OUT / 'v2_run2_best.onnx'
ONNX_DRIVE_LEGACY= DRIVE_OUT / 'best.onnx'
LOCAL_ONNX_PATH  = Path('/content/best.onnx')
CONTAINER_MODELS = Path('/content/container/models')
CONTAINER_MODELS.mkdir(parents=True, exist_ok=True)
dest = CONTAINER_MODELS / 'best.onnx'

# Priority: v2_run2_best.onnx > local best.onnx > best.onnx on Drive
if ONNX_DRIVE_R2.exists():
    shutil.copy(ONNX_DRIVE_R2, dest)
    print(f'✅ Copied v2_run2_best.onnx → {dest}')
elif LOCAL_ONNX_PATH.exists():
    shutil.copy(LOCAL_ONNX_PATH, dest)
    print(f'✅ Copied local best.onnx → {dest}')
elif ONNX_DRIVE_LEGACY.exists():
    shutil.copy(ONNX_DRIVE_LEGACY, dest)
    print(f'✅ Copied Drive best.onnx → {dest}')
else:
    print('⚠️  No ONNX file found. Run Section 6 or place best.onnx on Drive.')


⚠️  No ONNX file found. Run Section 6 or place best.onnx on Drive.


## Part B — Containerise the Inference

Create the full `container/` directory with all required source files.


### B1 · Directory skeleton


In [24]:
from pathlib import Path

ROOT = Path('/content')  # or Path('.') if running locally
C = ROOT / 'container'

for d in ['app', 'models']:
    (C / d).mkdir(parents=True, exist_ok=True)

print('container/ skeleton ready')


container/ skeleton ready


### B2 · STUDENT.json


In [25]:
import json
from pathlib import Path

C = Path('/content/container')

student = {
    'first_name': 'Ilaha',
    'last_name': 'Shafizada',
    'team': 'ilaha-shafizada',
    'model': {
        'framework': 'yolo26',
        'variant': 'yolo26m',
        'imgsz': 640,
        'epochs_total': 60,
        'tricks': ['larger_backbone', 'strong_augmentation', 'cos_lr', 'weight_decay', 'early_stopping']
    },
    'notes': 'Improved YOLO26 cat detector exported to ONNX and packaged for leaderboard inference.'
}

(C / 'STUDENT.json').write_text(json.dumps(student, indent=2))
print('✅ STUDENT.json written')


✅ STUDENT.json written


### B3 · requirements.txt


In [26]:
from pathlib import Path
C = Path('/content/container')

(C / 'requirements.txt').write_text(
    'onnxruntime==1.18.0\n'
    'numpy\n'
    'pillow\n'
    'opencv-python-headless\n'
)
print('✅ requirements.txt written')


✅ requirements.txt written


### B4 · Dockerfile


In [27]:
from pathlib import Path
C = Path('/content/container')

dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY container/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY container/app /app/app
COPY container/models /app/models
COPY container/STUDENT.json /app/STUDENT.json

ENTRYPOINT ["python", "/app/app/cli.py"]
'''

(C / 'Dockerfile').write_text(dockerfile)
print('✅ Dockerfile written')


✅ Dockerfile written


### B5 · app/\_\_init\_\_.py


In [28]:
from pathlib import Path
C = Path('/content/container')
(C / 'app' / '__init__.py').write_text('')
print('✅ app/__init__.py written (empty)')


✅ app/__init__.py written (empty)


### B6 · app/detector.py


In [29]:
import base64
from pathlib import Path
C = Path('/content/container')

_det_b64 = 'ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgb25ueHJ1bnRpbWUgYXMgb3J0CmZyb20gUElMIGltcG9ydCBJbWFnZQoKCmNsYXNzIENhdERldGVjdG9yOgogICAgQ0xBU1NfTkFNRVMgPSAoImNhdCIsKQoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsX3BhdGg6IHN0ciA9ICIvYXBwL21vZGVscy9iZXN0Lm9ubngiLAogICAgICAgIGltZ3N6OiBpbnQgPSA2NDAsCiAgICAgICAgY29uZl90aHJlc2hvbGQ6IGZsb2F0ID0gMC4yNSwKICAgICk6CiAgICAgICAgc2VsZi5pbWdzeiA9IGltZ3N6CiAgICAgICAgc2VsZi5jb25mX3RocmVzaG9sZCA9IGNvbmZfdGhyZXNob2xkCiAgICAgICAgc2VsZi5zZXNzaW9uID0gb3J0LkluZmVyZW5jZVNlc3Npb24oCiAgICAgICAgICAgIHN0cihtb2RlbF9wYXRoKSwgcHJvdmlkZXJzPVsiQ1BVRXhlY3V0aW9uUHJvdmlkZXIiXQogICAgICAgICkKICAgICAgICBzZWxmLl9pbnB1dF9uYW1lID0gc2VsZi5zZXNzaW9uLmdldF9pbnB1dHMoKVswXS5uYW1lCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBfbGV0dGVyYm94KHNlbGYsIGltZzogSW1hZ2UuSW1hZ2UpOgogICAgICAgICIiIlJlc2l6ZSArIHBhZCB0byAoaW1nc3osIGltZ3N6KSBwcmVzZXJ2aW5nIGFzcGVjdCByYXRpby4iIiIKICAgICAgICB3LCBoID0gaW1nLnNpemUKICAgICAgICBzY2FsZSA9IG1pbihzZWxmLmltZ3N6IC8gdywgc2VsZi5pbWdzeiAvIGgpCiAgICAgICAgbncsIG5oID0gaW50KHcgKiBzY2FsZSksIGludChoICogc2NhbGUpCiAgICAgICAgcmVzaXplZCA9IGltZy5yZXNpemUoKG53LCBuaCksIEltYWdlLkJJTElORUFSKQogICAgICAgIGNhbnZhcyA9IEltYWdlLm5ldygiUkdCIiwgKHNlbGYuaW1nc3osIHNlbGYuaW1nc3opLCAoMTE0LCAxMTQsIDExNCkpCiAgICAgICAgcGFkX3ggPSAoc2VsZi5pbWdzeiAtIG53KSAvLyAyCiAgICAgICAgcGFkX3kgPSAoc2VsZi5pbWdzeiAtIG5oKSAvLyAyCiAgICAgICAgY2FudmFzLnBhc3RlKHJlc2l6ZWQsIChwYWRfeCwgcGFkX3kpKQogICAgICAgIHJldHVybiBucC5hcnJheShjYW52YXMsIGR0eXBlPW5wLnVpbnQ4KSwgc2NhbGUsIChwYWRfeCwgcGFkX3kpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBwcmVkaWN0KHNlbGYsIGltYWdlX3BhdGg6IHN0cikgLT4gbGlzdFtkaWN0XToKICAgICAgICBpbWcgPSBJbWFnZS5vcGVuKGltYWdlX3BhdGgpLmNvbnZlcnQoIlJHQiIpCiAgICAgICAgb3JpZ193LCBvcmlnX2ggPSBpbWcuc2l6ZQoKICAgICAgICBhcnIsIHNjYWxlLCAocGFkX3gsIHBhZF95KSA9IHNlbGYuX2xldHRlcmJveChpbWcpCiAgICAgICAgeCA9IChhcnIuYXN0eXBlKG5wLmZsb2F0MzIpIC8gMjU1LjApLnRyYW5zcG9zZSgyLCAwLCAxKVtOb25lLCAuLi5dCgogICAgICAgIHJhdyA9IHNlbGYuc2Vzc2lvbi5ydW4oTm9uZSwge3NlbGYuX2lucHV0X25hbWU6IHh9KVswXQoKICAgICAgICAjIEhhbmRsZSAoMSwgMzAwLCA2KSBvciAoMzAwLCA2KQogICAgICAgIGlmIHJhdy5uZGltID09IDM6CiAgICAgICAgICAgIHJhdyA9IHJhd1swXQoKICAgICAgICByZXN1bHRzID0gW10KICAgICAgICBmb3Igcm93IGluIHJhdzoKICAgICAgICAgICAgeDEsIHkxLCB4MiwgeTIsIHNjb3JlLCBjbHMgPSByb3cKICAgICAgICAgICAgaWYgZmxvYXQoc2NvcmUpIDwgc2VsZi5jb25mX3RocmVzaG9sZDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgVW5kbyBsZXR0ZXJib3gKICAgICAgICAgICAgcngxID0gKGZsb2F0KHgxKSAtIHBhZF94KSAvIHNjYWxlCiAgICAgICAgICAgIHJ5MSA9IChmbG9hdCh5MSkgLSBwYWRfeSkgLyBzY2FsZQogICAgICAgICAgICByeDIgPSAoZmxvYXQoeDIpIC0gcGFkX3gpIC8gc2NhbGUKICAgICAgICAgICAgcnkyID0gKGZsb2F0KHkyKSAtIHBhZF95KSAvIHNjYWxlCiAgICAgICAgICAgICMgQ2xpcCB0byBpbWFnZSBib3VuZHMKICAgICAgICAgICAgcngxID0gbWF4KDAuMCwgbWluKG9yaWdfdywgcngxKSkKICAgICAgICAgICAgcnkxID0gbWF4KDAuMCwgbWluKG9yaWdfaCwgcnkxKSkKICAgICAgICAgICAgcngyID0gbWF4KDAuMCwgbWluKG9yaWdfdywgcngyKSkKICAgICAgICAgICAgcnkyID0gbWF4KDAuMCwgbWluKG9yaWdfaCwgcnkyKSkKICAgICAgICAgICAgIyBTa2lwIGRlZ2VuZXJhdGUgYm94ZXMKICAgICAgICAgICAgaWYgcngyIDw9IHJ4MSBvciByeTIgPD0gcnkxOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoewogICAgICAgICAgICAgICAgInhtaW4iOiByeDEsICJ5bWluIjogcnkxLCAieG1heCI6IHJ4MiwgInltYXgiOiByeTIsCiAgICAgICAgICAgICAgICAiY29uZmlkZW5jZSI6IGZsb2F0KHNjb3JlKSwKICAgICAgICAgICAgICAgICJjbGFzcyI6IHNlbGYuQ0xBU1NfTkFNRVNbaW50KGNscyldIGlmIGludChjbHMpIDwgbGVuKHNlbGYuQ0xBU1NfTkFNRVMpIGVsc2UgImNhdCIsCiAgICAgICAgICAgIH0pCiAgICAgICAgcmV0dXJuIHJlc3VsdHMK'
detector_src = base64.b64decode(_det_b64).decode()
(C / 'app' / 'detector.py').write_text(detector_src)
print('✅ app/detector.py written')
print(detector_src[:300])


✅ app/detector.py written
from pathlib import Path
import numpy as np
import onnxruntime as ort
from PIL import Image


class CatDetector:
    CLASS_NAMES = ("cat",)

    def __init__(
        self,
        model_path: str = "/app/models/best.onnx",
        imgsz: int = 640,
        conf_threshold: float = 0.25,
    ):
     


### B7 · app/cli.py


In [30]:
import base64
from pathlib import Path
C = Path('/content/container')

_cli_b64 = 'aW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gZGV0ZWN0b3IgaW1wb3J0IENhdERldGVjdG9yCgpTVFVERU5UX0pTT04gPSBQYXRoKCIvYXBwL1NUVURFTlQuanNvbiIpCklOUFVUX0RJUiAgICA9IFBhdGgoIi9kYXRhL2lucHV0IikKT1VUUFVUX0RJUiAgID0gUGF0aCgiL2RhdGEvb3V0cHV0IikKRVhURU5TSU9OUyAgID0geyIuanBnIiwgIi5qcGVnIiwgIi5wbmcifQoKCmRlZiBjbWRfaW5mbyhfYXJncyk6CiAgICBwcmludChTVFVERU5UX0pTT04ucmVhZF90ZXh0KCkpCgoKZGVmIGNtZF9wcmVkaWN0KF9hcmdzKToKICAgIE9VVFBVVF9ESVIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgY3N2X3BhdGggPSBPVVRQVVRfRElSIC8gInByZWRpY3Rpb25zLmNzdiIKCiAgICBkZXRlY3RvciA9IENhdERldGVjdG9yKCkKICAgIGltYWdlcyA9IHNvcnRlZCgKICAgICAgICBwIGZvciBwIGluIElOUFVUX0RJUi5yZ2xvYigiKiIpIGlmIHAuc3VmZml4Lmxvd2VyKCkgaW4gRVhURU5TSU9OUwogICAgKQoKICAgIHdpdGggY3N2X3BhdGgub3BlbigidyIsIG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGZoOgogICAgICAgIHdyaXRlciA9IGNzdi53cml0ZXIoZmgpCiAgICAgICAgd3JpdGVyLndyaXRlcm93KFsiaW1hZ2VfcGF0aCIsICJ4bWluIiwgInltaW4iLCAieG1heCIsICJ5bWF4IiwgImNvbmZpZGVuY2UiLCAiY2xhc3MiXSkKCiAgICAgICAgZm9yIGltZ19wYXRoIGluIGltYWdlczoKICAgICAgICAgICAgcmVsX3N0ciA9IGltZ19wYXRoLnJlbGF0aXZlX3RvKElOUFVUX0RJUikuYXNfcG9zaXgoKQogICAgICAgICAgICBkZXRzID0gZGV0ZWN0b3IucHJlZGljdChzdHIoaW1nX3BhdGgpKQogICAgICAgICAgICBpZiBub3QgZGV0czoKICAgICAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdyhbcmVsX3N0ciwgIiIsICIiLCAiIiwgIiIsICIiLCAiIl0pCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBmb3IgZCBpbiBkZXRzOgogICAgICAgICAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdyhbCiAgICAgICAgICAgICAgICAgICAgICAgIHJlbF9zdHIsCiAgICAgICAgICAgICAgICAgICAgICAgIGYie2RbJ3htaW4nXTouMmZ9IiwKICAgICAgICAgICAgICAgICAgICAgICAgZiJ7ZFsneW1pbiddOi4yZn0iLAogICAgICAgICAgICAgICAgICAgICAgICBmIntkWyd4bWF4J106LjJmfSIsCiAgICAgICAgICAgICAgICAgICAgICAgIGYie2RbJ3ltYXgnXTouMmZ9IiwKICAgICAgICAgICAgICAgICAgICAgICAgZiJ7ZFsnY29uZmlkZW5jZSddOi40Zn0iLAogICAgICAgICAgICAgICAgICAgICAgICBkWyJjbGFzcyJdLAogICAgICAgICAgICAgICAgICAgIF0pCgogICAgcHJpbnQoZiJQcm9jZXNzZWQge2xlbihpbWFnZXMpfSBpbWFnZShzKS4gUHJlZGljdGlvbnMgc2F2ZWQgdG8ge2Nzdl9wYXRofSIpCgoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJDYXQgRGV0ZWN0b3IgQ0xJIikKICAgIHN1YiA9IHBhcnNlci5hZGRfc3VicGFyc2VycyhkZXN0PSJjb21tYW5kIikKCiAgICBzdWIuYWRkX3BhcnNlcigiaW5mbyIsICAgIGhlbHA9IlByaW50IFNUVURFTlQuanNvbiIpCiAgICBzdWIuYWRkX3BhcnNlcigicHJlZGljdCIsIGhlbHA9IlJ1biBpbmZlcmVuY2Ugb24gL2RhdGEvaW5wdXQiKQoKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBpZiBhcmdzLmNvbW1hbmQgPT0gImluZm8iOgogICAgICAgIGNtZF9pbmZvKGFyZ3MpCiAgICBlbGlmIGFyZ3MuY29tbWFuZCA9PSAicHJlZGljdCI6CiAgICAgICAgY21kX3ByZWRpY3QoYXJncykKICAgIGVsc2U6CiAgICAgICAgcGFyc2VyLnByaW50X2hlbHAoKQogICAgICAgIHN5cy5leGl0KDEpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
cli_src = base64.b64decode(_cli_b64).decode()
(C / 'app' / 'cli.py').write_text(cli_src)
print('✅ app/cli.py written')
print(cli_src[:300])


✅ app/cli.py written
import argparse
import csv
import json
import sys
from pathlib import Path

from detector import CatDetector

STUDENT_JSON = Path("/app/STUDENT.json")
INPUT_DIR    = Path("/data/input")
OUTPUT_DIR   = Path("/data/output")
EXTENSIONS   = {".jpg", ".jpeg", ".png"}


def cmd_info(_args):
    print(STUD


### B8 · Validation — check all required files exist


In [31]:
import json
from pathlib import Path

C = Path('/content/container')

required_files = [
    'container/Dockerfile',
    'container/STUDENT.json',
    'container/requirements.txt',
    'container/app/__init__.py',
    'container/app/cli.py',
    'container/app/detector.py',
    'container/models/best.onnx',
]

print('=== container/ tree ===')
import subprocess
result = subprocess.run(['find', str(C), '-type', 'f'], capture_output=True, text=True)
for line in sorted(result.stdout.splitlines()):
    print(' ', line)

print('\n=== File check ===')
all_ok = True
for rel in required_files:
    p = Path('/content') / rel
    status = '✅' if p.exists() else '❌ MISSING'
    if not p.exists():
        all_ok = False
    print(f'  {status}  {rel}')

print('\n=== STUDENT.json contents ===')
sj = C / 'STUDENT.json'
if sj.exists():
    data = json.loads(sj.read_text())
    print(json.dumps(data, indent=2))
    print('\n✅ STUDENT.json is valid JSON')
else:
    print('❌ STUDENT.json missing')

print('\n=== Expected CSV header ===')
print('image_path,xmin,ymin,xmax,ymax,confidence,class')

if all_ok:
    print('\n✅ All required files present!')
else:
    print('\n⚠️  Some files missing — re-run cells above.')


=== container/ tree ===
  /content/container/Dockerfile
  /content/container/STUDENT.json
  /content/container/app/__init__.py
  /content/container/app/cli.py
  /content/container/app/detector.py
  /content/container/requirements.txt

=== File check ===
  ✅  container/Dockerfile
  ✅  container/STUDENT.json
  ✅  container/requirements.txt
  ✅  container/app/__init__.py
  ✅  container/app/cli.py
  ✅  container/app/detector.py
  ❌ MISSING  container/models/best.onnx

=== STUDENT.json contents ===
{
  "first_name": "Ilaha",
  "last_name": "Shafizada",
  "team": "ilaha-shafizada",
  "model": {
    "framework": "yolo26",
    "variant": "yolo26m",
    "imgsz": 640,
    "epochs_total": 60,
    "tricks": [
      "larger_backbone",
      "strong_augmentation",
      "cos_lr",
      "weight_decay",
      "early_stopping"
    ]
  },
  "notes": "Improved YOLO26 cat detector exported to ONNX and packaged for leaderboard inference."
}

✅ STUDENT.json is valid JSON

=== Expected CSV header ===
image_p

In [32]:
# ── Syntax check: compile both Python source files ───────────────────────
import subprocess, sys

files = [
    '/content/container/app/cli.py',
    '/content/container/app/detector.py',
]

result = subprocess.run(
    [sys.executable, '-m', 'py_compile'] + files,
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ py_compile passed — no syntax errors in cli.py or detector.py')
else:
    print('❌ Syntax error detected:')
    print(result.stderr)


✅ py_compile passed — no syntax errors in cli.py or detector.py


## 8 · Docker Build / Test / Push Commands

Run these in a terminal at the repository root (where `container/` lives).

### Build
```bash
docker build -t <dockerhub-username>/cat-detector:final -f container/Dockerfile .
```

### Test — info
```bash
docker run --rm <dockerhub-username>/cat-detector:final info
```
Expected: contents of `STUDENT.json` printed to stdout, exit code 0.

### Test — predict (Windows CMD / Anaconda Prompt)
```bat
docker run --rm ^
  -v "C:\absolute\path\to\test_images:/data/input:ro" ^
  -v "C:\absolute\path\to\output:/data/output" ^
  <dockerhub-username>/cat-detector:final predict
```

### Test — predict (Git Bash / macOS / Linux)
```bash
docker run --rm \
  -v "/c/absolute/path/to/test_images:/data/input:ro" \
  -v "/c/absolute/path/to/output:/data/output" \
  <dockerhub-username>/cat-detector:final predict
```

### Push
```bash
docker login
docker push <dockerhub-username>/cat-detector:final
```

---

## Root README.md — copy this block into your repository README

```markdown
## Image for leaderboard

docker pull <dockerhub-username>/cat-detector:final

Image: <dockerhub-username>/cat-detector:final  
Student: Ilaha Shafizada
```

---

## Final Submission Checklist

- [ ] At least 3 Week-2 techniques applied and documented
- [ ] Comparison table vs Week-1 baseline (no invented metrics)
- [ ] Best model exported to ONNX (`v2_run2_best.pt` preferred; `v2_run1_best.pt` fallback)
- [ ] ONNX sanity check shown (shape + inference)
- [ ] ONNX-vs-PyTorch comparison shown
- [ ] `container/STUDENT.json` valid JSON with correct identity
- [ ] `container/Dockerfile` builds cleanly (ONNX Runtime CPU only)
- [ ] `info` subcommand prints JSON, exit 0
- [ ] `predict` subcommand writes `/data/output/predictions.csv`
- [ ] CSV schema exactly: `image_path,xmin,ymin,xmax,ymax,confidence,class`
- [ ] No-detection rows written as `image.jpg,,,,,,`
- [ ] CSV verified on a local test folder
- [ ] Image pushed to public Docker Hub registry
- [ ] Pull command in repository root README
